In [ ]:
import numpy as np
import pandas as pd
import h5py
from pprint import pprint
from IPython.display import display
from typing import List, Dict, Any, Literal
import anndata
from anndata._io.h5ad import read_elem
from scipy.sparse import csr_matrix
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from tqdm.notebook import tqdm

In [ ]:
def detect_matrix_format(group) -> str:
    """Determine the matrix format based on the keys and structure of the group."""
    
    # If the group is an HDF5 dataset, check its shape and attributes.
    if isinstance(group, h5py.Dataset):
        # Check if it represents a multi-dimensional array (e.g., a dense matrix or NumPy array)
        if len(group.shape) >= 2:
            return "NumPy Array or Dense Matrix"
        else:
            return "1D Dataset"
    
    # Keys in the group
    keys = set(group.keys())
    
    # CSR or CSC Matrix Check
    if {'data', 'indices', 'indptr'}.issubset(keys):
        if 'shape' in group.attrs:
            return "CSR/CSC Matrix"
    
    # COO Matrix Check
    elif {'data', 'row', 'col'}.issubset(keys):
        return "COO Matrix"
    
    # Unknown format
    return "Unknown Format"

def generate_summary_report(data_dir: str) -> Dict[str, Any]:
    try:
        with h5py.File(data_dir, "r") as file:
            keys = file.keys()
            summary_report = {}
            
            for key in keys:
                group = file[key]
                
                if isinstance(group, h5py.Group):
                    group_keys = list(group.keys())
                    group_types = [type(group[k]) for k in group_keys]
                    summary_report[key] = {
                        "group_keys": group_keys,
                        "group_types": group_types,
                    }
                    
                    # Process the 'X' group
                    if key == 'X':
                        matrix_format = detect_matrix_format(group)
                        summary_report[key]["format"] = matrix_format
                        
                        # Report the shape directly for 'X'
                        if matrix_format in ["CSR/CSC Matrix", "COO Matrix"]:
                            if 'shape' in group.attrs:
                                summary_report[key]["shape"] = group.attrs['shape']
                            else:
                                # Fallback shape calculation (try to avoid if possible)
                                summary_report[key]["shape"] = (
                                    group["indptr"].shape[0] - 1,
                                    max(group["indices"]) + 1
                                )
                        elif matrix_format == "NumPy Array or Dense Matrix":
                            summary_report[key]["shape"] = group.shape  # Directly use group.shape

                    # Process the 'layers' group
                    if key == 'layers': 
                        layer_dict = {}
                        for k in file["layers"].keys():
                            sub_group = file[key][k]
                            
                            matrix_format = detect_matrix_format(sub_group)
                            layer_dict[k] = {
                                "format": matrix_format,
                            }
                            
                            # Directly use the shape if available
                            if 'shape' in sub_group.attrs:
                                layer_dict[k]["shape"] = sub_group.attrs['shape']
                            else:
                                if matrix_format == "NumPy Array or Dense Matrix":
                                    layer_dict[k]["shape"] = sub_group.shape
                                elif matrix_format == "CSR/CSC Matrix":
                                    # Only calculate shape manually if absolutely necessary
                                    layer_dict[k]["shape"] = (
                                        sub_group["indptr"].shape[0] - 1,
                                        max(sub_group["indices"]) + 1
                                    )
                        
                        summary_report[key]["sublayer_info"] = layer_dict
                    
                elif isinstance(group, h5py.Dataset):
                    dtype = group.dtype if hasattr(group, "dtype") else None
                    summary_report[key] = {"shape": group.shape, "dtype": dtype}
        
        return summary_report
    
    except Exception as e:
        print(f"Error processing file: {e}")
        return {}

# Example usage:
# report = generate_summary_report("your_file_path_here.h5")
# pprint(report)


In [ ]:
#data_dir = '/nfs/team298/ar32/repos/H5py_anndata_checker/dummy_data/test1.h5ad'
#data_dir = '/nfs/team298/ar32/repos/H5py_anndata_checker/dummy_data/test2.h5ad'
data_dir = '/nfs/team298/ar32/repos/H5py_anndata_checker/dummy_data/test4.h5ad'

In [ ]:
def generate_anndata_report(
    data_dir: str,
):
    report = generate_summary_report(data_dir)
    print(f"\033[1mAnndata object checking: {data_dir}\033[0m\n")

    # Print the summary report
    for key, value in report.items():
        print(f"\033[1mPartition: {key}\033[0m\n")

        if "group_keys" in value:
            group_info = {
                "Group Keys": value['group_keys'],
                "Group Types": value['group_types']
            }
            pprint(group_info, indent=4, width=80, compact=True)

            if key == "X":
                matrix_info = {
                    "Matrix Format": value['format'],
                    "Shape": value.get("shape"),
                }

                # Extract and add number of cells and features from shape
                if "shape" in value:
                    num_cells = value["shape"][0]
                    num_features = value["shape"][1]
                    matrix_info["Number of cells"] = num_cells
                    matrix_info["Number of features"] = num_features

                pprint(matrix_info, indent=4, width=80, compact=True)

            elif key == "layers":
                print("\nSub layer information:\n")
                for k, v in value["sublayer_info"].items():
                    print(f"{k}:")
                    layer_info = {
                        "Matrix Format": v['format'],
                    }

                    # Handle shape and extract number of cells and features if shape is present
                    if "shape" in v:
                        layer_info["Shape"] = v["shape"]
                        num_cells = v["shape"][0]
                        num_features = v["shape"][1]
                        layer_info["Number of cells"] = num_cells
                        layer_info["Number of features"] = num_features

                    if "Cell_num" in v:
                        layer_info["Number of cells"] = v["Cell_num"]
                    if "Feat_num" in v:
                        layer_info["Number of features"] = v["Feat_num"]

                    pprint(layer_info, indent=6, width=80, compact=True)
                    print("")  # Empty line for readability

        else:
            dataset_info = {
                "Shape": value['shape'],
                "Dtype": value['dtype']
            }
            pprint(dataset_info, indent=4, width=80, compact=True)

        print("---------------------------\n")

In [ ]:
generate_anndata_report(data_dir)

In [ ]:
#def inspect_column_categories(
#    data_dir: str,
#    dataframe: Literal['obs', 'var'],
#    columns: List[str],
#) -> pd.DataFrame:
#    
#    # Initialize an empty dictionary to store decoded categories
#    decoded_data: Dict[str, List[Any]] = {col: [] for col in columns}
#    
#    with h5py.File(data_dir, "r") as file:
#        # Iterate over columns and decode categories
#        for col in columns:
#            decoded_data[col] = [
#                value.decode() for value in file[dataframe][col]["categories"]
#            ]
#    
#    # Find the maximum length among all categories
#    max_length = max(len(decoded_data[col]) for col in columns)
#    
#    # Pad the lists in the dictionary with empty strings if needed
#    for col in columns:
#        decoded_data[col] += [""] * (max_length - len(decoded_data[col]))
#    
#    # Create a DataFrame from the decoded data
#    df = pd.DataFrame(decoded_data)
#    
#    # Update column names with ' unique values'
#    df.columns = [f"{col} unique values" for col in columns]
#    
#    
#    # Return the DataFrame to be used in memory
#    return df

import h5py
import pandas as pd
from typing import List, Dict, Literal, Any

def inspect_column_categories(
    data_dir: str,
    dataframe: Literal['obs', 'var'],
    columns: List[str],
) -> pd.DataFrame:
    """
    Inspects the categories or unique values of specified columns in an HDF5 file and returns a DataFrame with the decoded values.

    Args:
        data_dir (str): The path to the HDF5 file.
        dataframe (Literal['obs', 'var']): The type of the dataframe to inspect.
        columns (List[str]): A list of column names to inspect.

    Returns:
        pd.DataFrame: A DataFrame where each column contains the unique category values or unique values of the specified columns.
    """
    
    # Initialize an empty dictionary to store decoded categories or unique values
    decoded_data: Dict[str, List[Any]] = {col: [] for col in columns}
    
    with h5py.File(data_dir, "r") as file:
        # Check if the specified dataframe exists in the file
        if dataframe not in file:
            raise ValueError(f"DataFrame type '{dataframe}' not found in file.")
        
        df_group = file[dataframe]
        
        # Iterate over columns and decode categories or gather unique values
        for col in columns:
            if col in df_group:
                data = df_group[col]
                
                if "categories" in data:
                    # Decode category values if the column has categories
                    decoded_data[col] = [value.decode() for value in data["categories"]]
                else:
                    # For non-categorical columns, gather unique values
                    # Note: Adjust depending on actual data storage
                    decoded_data[col] = list(set(data))
            else:
                # Handle the case where the column is not found
                decoded_data[col] = ["Column not found"] 

    # Find the maximum length among all categories or unique values
    max_length = max(len(decoded_data[col]) for col in columns)
    
    # Pad the lists in the dictionary with empty strings if needed
    for col in columns:
        if len(decoded_data[col]) < max_length:
            decoded_data[col] += [""] * (max_length - len(decoded_data[col]))
    
    # Create a DataFrame from the decoded data
    df = pd.DataFrame(decoded_data)
    
    # Update column names with ' unique values'
    df.columns = [f"{col} unique values" for col in columns]
    
    return df


In [ ]:
dataframe = 'obs'
columns = [
    'anno_LVL1', 
    'anno_LVL2',
    'study',
    ]

#dataframe = 'var' # choose obs or var 
#columns = [
#    'high_var', 
#    ]

# Display the DataFrame
print(
    f"\033[1mDataFrame output to see all unique values for each column of interest in {dataframe}:\033[0m\n"
)
inspect_column_categories(data_dir, dataframe, columns)

In [ ]:
#dataframe = 'obs'
#columns = [
#    'anno_LVL1', 
#    'anno_LVL2',
#    'study',
#    ]

dataframe = 'var' # choose obs or var 
columns = [
    'high_var', 
    ]

# Display the DataFrame
print(
    f"\033[1mDataFrame output to see all unique values for each column of interest in {dataframe}:\033[0m\n"
)
inspect_column_categories(data_dir, dataframe, columns)

In [ ]:
def extract_dataframe(
    file,
    dataframe: Literal['obs', 'var'],
    filter_dict: Dict[str, List[str]],
    additional_cols: List[str],
    filter_method: Literal['intersection', 'union'] = 'intersection'
):
    # Retrieve and decode index
    original_index_values = np.vectorize(lambda x: x.decode("utf-8"))(
        np.array(file[dataframe]["_index"], dtype=object)
    )
    
    # Create a DataFrame with both original index values and their positions
    original_index_df = pd.DataFrame({
        "Original_index_value": original_index_values,
        "Original_index_position": np.arange(len(original_index_values))
    }, index=original_index_values)

    # Initialize a list to keep DataFrames for filtering
    filtered_dfs = []

    # Apply filtering for each column specified in filter_dict
    for filter_column, filter_values in filter_dict.items():
        col_data = pd.DataFrame(read_elem(file[dataframe][filter_column])).astype(str)
        col_data = col_data[col_data.iloc[:, 0].isin(filter_values)]
        missing_values = set(filter_values) - set(col_data.iloc[:, 0])

        if missing_values:
            raise ValueError(f"Missing values in filter_column '{filter_column}': {missing_values}")

        col_data.rename(columns={0: filter_column}, inplace=True)
        col_data.index = original_index_values[col_data.index]
        
        # Add the original index value and position columns to col_data
        col_data.insert(0, "Original_index_value", col_data.index)
        col_data.insert(1, "Original_index_position", original_index_df.loc[col_data.index, "Original_index_position"])

        filtered_dfs.append(col_data)

    # Combine filtered DataFrames based on filter_method
    if filter_method == 'intersection':
        # Intersect all filtered DataFrames
        common_index = set(filtered_dfs[0].index)
        for df in filtered_dfs[1:]:
            common_index &= set(df.index)
        common_index = list(common_index)
    elif filter_method == 'union':
        # Union of all filtered DataFrames
        common_index = set()
        for df in filtered_dfs:
            common_index |= set(df.index)
        common_index = list(common_index)
    else:
        raise ValueError("Invalid filter_method. Choose either 'intersection' or 'union'.")

    # Combine columns into a single DataFrame
    combined_cols = {df.columns[2]: df[df.index.isin(common_index)] for df in filtered_dfs}

    # Add additional columns
    if additional_cols:
        for col in additional_cols:
            col_data = pd.DataFrame(read_elem(file[dataframe][col]))
            col_data.rename(columns={0: col}, inplace=True)
            col_data.index = original_index_values
            col_data = col_data[col_data.index.isin(common_index)]
            combined_cols[col] = col_data
    else:
        df = pd.DataFrame(read_elem(file[dataframe]))
        df.index = original_index_values
        df = df[df.index.isin(common_index)]
        df["Original_index_value"] = df.index
        df["Original_index_position"] = original_index_df.loc[df.index, "Original_index_position"]
        for col in list(df.columns):
            combined_cols[col] = df[[col]]

    # Final DataFrame assembly
    out_df = pd.concat(combined_cols.values(), axis=1)
    original_col_order = ["Original_index_position", "Original_index_value"] + list(file[dataframe].attrs.get("column-order", []))
    original_col_order = [item for item in original_col_order if item in out_df.columns]
    out_df = out_df.loc[:, ~out_df.columns.duplicated()]
    out_df = out_df.filter(items=original_col_order)
    out_df.reset_index(drop=True, inplace=True)
    
    return out_df

def create_dataframe_subset(
    data_dir: str,
    dataframe: Literal['obs', 'var'],
    filter_dict: Dict[str, List[str]],  # Dictionary to specify filter columns and their values
    additional_cols: List[str],
    filter_method: Literal['intersection', 'union'] = 'intersection'  # Method to apply filtering
) -> pd.DataFrame:
    
    with h5py.File(data_dir, "r") as file:
        
        subset_dataframe = extract_dataframe(file, dataframe, filter_dict, additional_cols, filter_method)
        
    return subset_dataframe



In [ ]:
dataframe = 'obs'

filter_dict = {
    
    'anno_LVL2' : ['macrophage','cardiomyocyte',],
    #'anno_LVL1' : ['epithelial']
    'anno_LVL1' : ['stromal']
}

additional_cols_keep = [
    'anno_LVL1',
    'biological_unit',
    ]

filter_method = 'intersection'
#filter_method = 'union'


print(
        f"\033[1mDataFrame output of {dataframe} subset by columns {list(filter_dict.keys())} by {filter_method} for values of interest:\033[0m\n"
    )
create_dataframe_subset(data_dir, dataframe, filter_dict, additional_cols_keep, filter_method)

In [ ]:
dataframe = 'obs'

filter_dict = {
    
    'anno_LVL2' : ['macrophage','cardiomyocyte',],
    #'anno_LVL1' : ['epithelial']
    'anno_LVL1' : ['stromal']
}

additional_cols_keep = []

filter_method = 'intersection'
#filter_method = 'union'


print(
        f"\033[1mDataFrame output of {dataframe} subset by columns {list(filter_dict.keys())} by {filter_method} for values of interest:\033[0m\n"
    )
create_dataframe_subset(data_dir, dataframe, filter_dict, additional_cols_keep, filter_method)

In [ ]:
dataframe = 'var'

filter_dict = {
    
    'high_var' : ['yes'],
}

additional_cols_keep = []

filter_method = 'intersection'
#filter_method = 'union'


print(
        f"\033[1mDataFrame output of {dataframe} subset by columns {list(filter_dict.keys())} by {filter_method} for values of interest:\033[0m\n"
    )
create_dataframe_subset(data_dir, dataframe, filter_dict, additional_cols_keep, filter_method)

In [ ]:
import h5py
import numpy as np
from scipy.sparse import csr_matrix
from tqdm import tqdm
from typing import List, Dict, Literal, Tuple

In [ ]:
def sparse_grab_filtered_values(
    rows_to_load: List[int],
    cols_to_load: List[int],
    data_dset: np.ndarray,
    indices_dset: np.ndarray,
    indptr_dset: np.ndarray,
    description: str,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Extracts data for specific rows and columns from a sparse matrix (CSR/CSC format).
    """
    selected_data = []
    selected_indices = []
    selected_indptr = [0]

    # Mapping from original to filtered indices
    row_map = {orig_idx: new_idx for new_idx, orig_idx in enumerate(rows_to_load)}
    col_map = {orig_idx: new_idx for new_idx, orig_idx in enumerate(cols_to_load)}

    # Process each row
    for row_idx in tqdm(
        rows_to_load,
        desc=f"Processing Rows and Columns for {description}",
        unit="row",
        position=0,
        leave=True,
    ):
        start_idx = indptr_dset[row_idx]
        end_idx = indptr_dset[row_idx + 1]

        # Filter the indices and data for the columns we are interested in
        row_indices = indices_dset[start_idx:end_idx]
        row_data = data_dset[start_idx:end_idx]

        # Adjust indices to match filtered columns
        filtered_indices = [col_map.get(idx, -1) for idx in row_indices if idx in col_map]
        filtered_data = [row_data[i] for i in range(len(row_indices)) if row_indices[i] in col_map]

        # Add the filtered data to the list
        selected_data.extend(filtered_data)
        selected_indices.extend(filtered_indices)
        selected_indptr.append(selected_indptr[-1] + len(filtered_data))

    selected_data = np.array(selected_data)
    selected_indices = np.array(selected_indices)
    selected_indptr = np.array(selected_indptr)

    return selected_data, selected_indices, selected_indptr






def create_anndata_subset(
    data_dir: str,
    obs_filter_dict: Dict[str, List[str]],
    obs_additional_cols_keep: List[str],
    obs_filter_method: Literal['intersection', 'union'],
    var_filter_dict: Dict[str, List[str]],
    var_additional_cols_keep: List[str],
    var_filter_method: Literal['intersection', 'union'],
    filter_layers: List[str],
    filter_obsm: List[str],
    filter_obsp: List[str],
    filter_varm: List[str],
    filter_varp: List[str],
    filter_uns: List[str],
    keep_layers: bool = False,
    keep_obsm: bool = False,
    keep_obsp: bool = False,
    keep_varm: bool = False,
    keep_varp: bool = False,
    keep_uns: bool = False,
):
    
    with h5py.File(data_dir, "r") as file:
        
        if obs_filter_dict:
            obs_dataframe = extract_dataframe(file, "obs", obs_filter_dict, obs_additional_cols_keep, obs_filter_method)
            
            # List of row positions to load
            obs_rows_to_load = obs_dataframe["Original_index_position"].values
            #display(obs_rows_to_load)
        
        if var_filter_dict:
            var_dataframe = extract_dataframe(file, "var", var_filter_dict, var_additional_cols_keep, var_filter_method)
        
            # List of row positions to load
            var_cols_to_load = var_dataframe["Original_index_position"].values
            #display(var_rows_to_load)
            
        
        matrix_format = detect_matrix_format(file["X"])
        
        if matrix_format in ["CSR/CSC Matrix", "COO Matrix"]: # first test with csr, check later for csc and coo
            
            # Assign variables to query
            data_dset = file["X"]["data"]
            indices_dset = file["X"]["indices"]
            indptr_dset = file["X"]["indptr"]

            # Extract filtered data, indices, and indptr
            filtered_data, filtered_indices, filtered_indptr = sparse_grab_filtered_values(
                obs_rows_to_load,
                var_cols_to_load,
                data_dset,
                indices_dset,
                indptr_dset,
                "filtered data"
            )

            num_filtered_rows = len(obs_rows_to_load)
            num_filtered_cols = len(var_cols_to_load)

            # Create the subset matrix
            print(
                        "Constructing data into csr_matrix format:  \U0001F527",
                        flush=True,
                    )
            subset_matrix = csr_matrix(
                (filtered_data, filtered_indices, filtered_indptr),
                shape=(num_filtered_rows, num_filtered_cols),
                dtype=data_dset.dtype
            )
            print("Construction complete \u2705")
        
        
        # requires testing
        elif matrix_format == "NumPy Array or Dense Matrix":
            subset_matrix = file["X"][obs_rows_to_load, :][:, var_cols_to_load]
        
        obs_dataframe.index = obs_dataframe['Original_index_value'].copy()
        obs_dataframe.index.name = None
        
        var_dataframe.index = var_dataframe['Original_index_value'].copy()
        var_dataframe.index.name = None
        
        if keep_layers:
            if not filter_layers:
                layers = {}
                for x in file["layers"].keys():
                    # Assign variables to query
                    data_dset = file["layers"][x]["data"]
                    indices_dset = file["layers"][x]["indices"]
                    indptr_dset = file["layers"][x]["indptr"]

                    name = f"layer {x} data"

                    (
                        selected_rows_data,
                        selected_rows_indices,
                        selected_rows_indptr,
                    ) = sparse_grab_filtered_values(
                        rows_to_load, data_dset, indices_dset, indptr_dset, name
                    )

                    # Create csr_matrix directly from NumPy arrays
                    print(
                        "Constructing data into csr_matrix format:  \U0001F527",
                        flush=True,
                    )
                    layers[x] = csr_matrix(
                        (
                            selected_rows_data,
                            selected_rows_indices,
                            selected_rows_indptr,
                        ),
                        shape=(len(rows_to_load), num_columns),
                        dtype=file["layers"][x]["data"].dtype,
                    )
                    print("Construction complete \u2705")

            else:
                layers = {}
                for x in (
                    value for value in filter_layers if value in file["layers"].keys()
                ):
                    # Assign variables to query
                    data_dset = file["layers"][x]["data"]
                    indices_dset = file["layers"][x]["indices"]
                    indptr_dset = file["layers"][x]["indptr"]

                    name = f"layer {x} data"

                    (
                        selected_rows_data,
                        selected_rows_indices,
                        selected_rows_indptr,
                    ) = sparse_grab_filtered_values(
                        obs_rows_to_load, var_cols_to_load, data_dset, indices_dset, indptr_dset, name
                    )

                    # Create csr_matrix directly from NumPy arrays
                    print(
                        "Constructing data into csr_matrix format:  \U0001F527",
                        flush=True,
                    )
                    layers[x] = csr_matrix(
                        (
                            selected_rows_data,
                            selected_rows_indices,
                            selected_rows_indptr,
                        ),
                        shape=(len(obs_rows_to_load), len(var_cols_to_load)),
                        dtype=file["layers"][x]["data"].dtype,
                    )
                    print("Construction complete \u2705")

        else:
            layers = None

        if keep_obsm:
            if not filter_obsm:
                obsm = {
                    x: anndata._io.h5ad.read_elem(file["obsm"][x])[obs_rows_to_load]
                    for x in file["obsm"].keys()
                }
            else:
                obsm = {
                    x: anndata._io.h5ad.read_elem(file["obsm"][x])[obs_rows_to_load]
                    for x in filter_obsm
                    if x in file["obsm"].keys()
                }
        else:
            obsm = None

        if keep_obsp:
            if not filter_obsp:
                obsp = {
                    x: anndata._io.h5ad.read_elem(file["obsp"][x])[obs_rows_to_load][
                        :, obs_rows_to_load
                    ]
                    for x in file["obsp"].keys()
                }
            else:
                obsp = {
                    x: anndata._io.h5ad.read_elem(file["obsp"][x])[obs_rows_to_load][
                        :, obs_rows_to_load
                    ]
                    for x in filter_obsp
                    if x in file["obsp"].keys()
                }
        else:
            obsp = None

        if keep_varm:
            if not filter_varm:
                varm = {
                    x: anndata._io.h5ad.read_elem(file["varm"])[var_cols_to_load]
                    for x in file["varm"].keys()
                }
            else:
                varm = {
                    x: anndata._io.h5ad.read_elem(file["varm"][x])[var_cols_to_load]
                    for x in filter_varm
                    if x in file["varm"].keys()
                }
        else:
            varm = None

        if keep_varp:
            if not filter_varp:
                varp = {
                    x: anndata._io.h5ad.read_elem(file["varp"][x])[var_cols_to_load][
                        :, var_cols_to_load
                    ]
                    for x in file["varp"].keys()
                }
            else:
                varp = {
                    x: anndata._io.h5ad.read_elem(file["varp"][x])[var_cols_to_load][
                        :, var_cols_to_load
                    ]
                    for x in filter_varp
                    if x in file["varp"].keys()
                }
        else:
            varp = None

        if keep_uns:
            if not filter_uns:
                uns = anndata._io.h5ad.read_elem(file["uns"])
            else:
                uns = {
                    x: anndata._io.h5ad.read_elem(file["uns"][x])
                    for x in filter_uns
                    if x in file["uns"].keys()
                }
        else:
            uns = None

        adata = anndata.AnnData(
            X=subset_matrix,
            obs=obs_dataframe,
            var=var_dataframe,
            layers=layers,
            obsm=obsm,
            obsp=obsp,
            varm=varm,
            varp=varp,
            uns=uns,
        )
        
        return adata

In [ ]:
input_settings = {
    'data_dir' : data_dir,
    'obs_filter_dict' : {
                        'anno_LVL2' : ['macrophage','cardiomyocyte',],
                        #'anno_LVL1' : ['epithelial']
                        'anno_LVL1' : ['stromal']
                        },
    'obs_additional_cols_keep' : [],
    'obs_filter_method' : 'intersection',
    'var_filter_dict' : {
                        'high_var' : ['yes']
                        },
    'var_additional_cols_keep' : [],
    'var_filter_method' : 'intersection',
    
    'filter_layers' : [],
    'filter_obsm' : [],
    'filter_obsp' : [],
    'filter_varm' : [],
    'filter_varp' : [],
    'filter_uns' : [],
    
    'keep_layers' : False,
    'keep_obsm' : False,
    'keep_obsp' : False,
    'keep_varm' : False,
    'keep_varp' : False,
    'keep_uns' : False,
    
}

adata = create_anndata_subset(**input_settings)

print("")
print("")
print("\033[1mSubset anndata object generated successfully\033[0m\n")
print("\033[1m" + "Anndata whole preview:" + "\033[0m")
display(adata)
print("")
print("")
print("\033[1mQuick view of the anndata object generated\033[0m\n")
print(f"Overall shape: {adata.shape}")
print(f"Min count: {adata.X.min()}")
print(f"Max count: {adata.X.max()}")
print("")
print("\033[1m" + "obs preview:" + "\033[0m")
display(adata.obs)
print("")
print("\033[1m" + "var preview:" + "\033[0m")
display(adata.var)
print("")